In [8]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, DataCollatorWithPadding
import torch.nn as nn
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from transformers import ModernBertConfig
from tqdm import tqdm
from sklearn.metrics import classification_report






In [9]:
df_train = pd.read_csv("scratch/4641_data/train_dataset.csv")
print(df_train["text_length"].max())
df_train = df_train[["text", "sentiment", "bucket"]]
df_train["sentiment"] = df_train["sentiment"].map({"negative": 0, "neutral": 1, "positive": 2})

df_val = pd.read_csv("scratch/4641_data/val_dataset.csv")
df_val = df_val[["text", "sentiment", "bucket"]]
df_val["sentiment"] = df_val["sentiment"].map({"negative": 0, "neutral": 1, "positive": 2})

df_test = pd.read_csv("scratch/4641_data/test_dataset.csv")
df_test = df_test[["text", "sentiment", "bucket"]]
df_test["sentiment"] = df_test["sentiment"].map({"negative": 0, "neutral": 1, "positive": 2})


12930


In [10]:
class SentimentDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=3000):
        self.bodies = data["text"].values
        self.labels = data["sentiment"].values
        self.buckets = data["bucket"].values if "bucket" in data.columns else None
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.bodies)
    def __getitem__(self, idx):
        encoded = self.tokenizer(str(self.bodies[idx]), truncation=True, padding=False, max_length = self.max_length, return_tensors = "pt")
        return {"input_ids": encoded["input_ids"].squeeze(), "attention_mask": encoded["attention_mask"].squeeze(), "label": torch.tensor(self.labels[idx])}

In [11]:
num_classes = 3

In [12]:
class bertSentimentAnalyzer(nn.Module):
    def __init__ (self, num_classes=num_classes, freeze_base=True):
        super().__init__()
        self.base = bert_model
        self.dropout = nn.Dropout(0.3)
        self.linear1 = nn.Linear(self.base.config.hidden_size, num_classes)
        if freeze_base:
            for param in self.base.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        out = self.base(input_ids=input_ids, attention_mask=attention_mask)
        out = out.last_hidden_state[: , 0, :]
        out = self.dropout(out)
        out = self.linear1(out)
        return out

In [13]:
def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(loader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy, all_preds, all_labels

In [ ]:

tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
config = ModernBertConfig.from_pretrained("answerdotai/ModernBERT-base", reference_compile=False)


buckets = ["super_short", "short", "medium", "long"]
num_classes = 3
for bucket in buckets:
    print(f"Running bucket {bucket}")
    print()
    bucket_train = df_train[df_train["bucket"] == bucket]
    bucket_val = df_val[df_val["bucket"] == bucket]
    bucket_test = df_test[df_test["bucket"] == bucket]
    
    print(len(bucket_train))
    print(len(bucket_val))
    print(len(bucket_test))
    
    
    train_data = SentimentDataset(bucket_train, tokenizer)
    val_data = SentimentDataset(bucket_val, tokenizer)
    test_data = SentimentDataset(bucket_test, tokenizer)
    
    collator = DataCollatorWithPadding(tokenizer=tokenizer)
    train_loader = DataLoader(train_data, batch_size=16, shuffle=True, collate_fn=collator)
    val_loader = DataLoader(val_data, batch_size=16, collate_fn=collator)
    test_loader = DataLoader(test_data, batch_size=16, collate_fn=collator)
    
    
    bert_model = AutoModel.from_pretrained("answerdotai/ModernBERT-base", config=config)
    model = bertSentimentAnalyzer(freeze_base=False)
    criterion = nn.CrossEntropyLoss()
    #optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    num_epochs = 10
    training_steps = num_epochs * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*training_steps),num_training_steps=training_steps)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    #print(device)
    model.to(device)
    print(device)

    best_val_loss = float("inf")

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")

        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scheduler, criterion, device)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_model.pt")
            print("Saved best model!")


    print(f"Evaluating bucket {bucket}")
    model.load_state_dict(torch.load("best_model.pt"))
    test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, device)
    print(f"\nTest Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")


    print(classification_report(test_labels, test_preds, target_names=["negative", "neutral", "positive"]))
    print(f"End of bucket {bucket}")
    print()


Running bucket super_short

14000
2000
4000
cuda

Epoch 1/10


Evaluating: 100%|██████████| 125/125 [00:01<00:00, 80.00it/s]


Train Loss: 0.8946 | Train Acc: 0.5746
Val Loss:   0.6502 | Val Acc:   0.7425
Saved best model!

Epoch 2/10


Evaluating: 100%|██████████| 125/125 [00:01<00:00, 80.46it/s]


Train Loss: 0.5615 | Train Acc: 0.7734
Val Loss:   0.5383 | Val Acc:   0.7815
Saved best model!

Epoch 3/10


...

Train Loss: 0.0000 | Train Acc: 1.0000
Val Loss:   0.6041 | Val Acc:   0.9211

Epoch 10/10


Evaluating: 100%|██████████| 63/63 [00:04<00:00, 14.61it/s]


Train Loss: 0.0000 | Train Acc: 1.0000
Val Loss:   0.6072 | Val Acc:   0.9221
Evaluating bucket long


Evaluating: 100%|██████████| 125/125 [00:08<00:00, 14.11it/s]


Test Loss: 0.2996 | Test Acc: 0.8914
              precision    recall  f1-score   support

    negative       0.91      0.91      0.91       733
     neutral       0.86      0.81      0.83       533
    positive       0.90      0.93      0.91       733

    accuracy                           0.89      1999
   macro avg       0.89      0.88      0.89      1999
weighted avg       0.89      0.89      0.89      1999

End of bucket long

